In [1]:
# ============================================================
# KHỞI TẠO: Import module & nạp dữ liệu vital
# ============================================================

import pandas as pd
import numpy as np

from data_quality import (
    audit,
    check_overview,
    check_plausibility,
    check_distribution,
    check_hidden_issues,
    check_logic,
)

# Đường dẫn (tính từ data/quality_checks/)
vital_input   = "../processed/vital_clean.csv"
patient_input = "../processed/patient_clean.csv"

df_vital = pd.read_csv(vital_input)
df_patient = pd.read_csv(patient_input)

print("Đã nạp vital_clean.csv  :", df_vital.shape)
print("Đã nạp patient_clean.csv:", df_patient.shape)
print("\n5 dòng đầu tiên của vital:")
display(df_vital.head())

Đã nạp vital_clean.csv  : (2240, 10)
Đã nạp patient_clean.csv: (2326, 5)

5 dòng đầu tiên của vital:


,patientunitstayid,heartrate_mean,heartrate_min,heartrate_max,systemicmean_max,sao2_mean,sao2_min,respiration_mean,respiration_max,temperature_mean
0,141764,106.652174,90.0,138.0,NaN,NaN,NaN,NaN,NaN,NaN
1,141765,83.190972,72.0,116.0,NaN,96.608696,95.0,24.782609,39.0,NaN
2,143870,45.449367,40.0,55.0,79.0,96.284810,80.0,67.031646,86.0,NaN
3,144815,83.800000,68.0,118.0,NaN,98.630682,96.0,16.927273,42.0,NaN
4,145427,65.457983,49.0,85.0,NaN,95.761905,89.0,16.349206,41.0,NaN


In [2]:
# ============================================================
# KIỂM TRA CHUNG: dùng module data_quality
# ============================================================
# vital_clean.csv KHÔNG có cột nhãn mortality (đã tách riêng),
# nên truyền label_col=None để bỏ qua phần mất cân bằng nhãn.
#
# Ngưỡng plausibility cho 9 đặc trưng sinh hiệu:
#   - heartrate: 20–250 bpm
#   - systemicmean: 20–200 mmHg
#   - sao2: 50–100 %
#   - respiration: 5–80 lần/phút
#   - temperature: 30–45 °C

VITAL_RULES = {
    "heartrate_mean":     (20, 250, "bpm"),
    "heartrate_min":      (20, 250, "bpm"),
    "heartrate_max":      (20, 250, "bpm"),
    "systemicmean_max":   (20, 200, "mmHg"),
    "sao2_mean":          (50, 100, "%"),
    "sao2_min":           (50, 100, "%"),
    "respiration_mean":   (5, 80,  "lần/phút"),
    "respiration_max":    (5, 80,  "lần/phút"),
    "temperature_mean":   (30, 45, "°C"),
}

audit(
    df_vital,
    name="vital_clean.csv",
    label_col=None,
    plausibility_rules=VITAL_RULES,
)


TỔNG QUAN: vital_clean.csv
Kích thước : 2,240 dòng × 10 cột
Bộ nhớ     : 175.1 KB

Kiểu dữ liệu:
patientunitstayid      int64
heartrate_mean       float64
heartrate_min        float64
heartrate_max        float64
systemicmean_max     float64
sao2_mean            float64
sao2_min             float64
respiration_mean     float64
respiration_max      float64
temperature_mean     float64

Missing values:
                  missing    pct
heartrate_mean         11   0.49
heartrate_min          11   0.49
heartrate_max          11   0.49
systemicmean_max     1855  82.81
sao2_mean              36   1.61
sao2_min               36   1.61
respiration_mean      205   9.15
respiration_max       205   9.15
temperature_mean     2095  93.53

Số dòng trùng lặp hoàn toàn: 0

KIỂM TRA PLAUSIBILITY (ngưỡng sinh lý)
✅ heartrate_mean  | min=     37.00 max=    147.53 | ngoài [20, 250] (bpm): 0 dòng
⚠️ heartrate_min   | min=      2.00 max=    127.00 | ngoài [20, 250] (bpm): 10 dòng
✅ heartrate_max   | min=   

In [3]:
# ============================================================
# KIỂM TRA RIÊNG: Khóa chính
# ============================================================
# Mỗi patientunitstayid chỉ được xuất hiện đúng 1 lần.

print("=" * 60)
print("KIỂM TRA RIÊNG: KHÓA CHÍNH")
print("=" * 60)

n_rows   = len(df_vital)
n_unique = df_vital["patientunitstayid"].nunique()
n_dup    = n_rows - n_unique

print(f"\nSố dòng              : {n_rows:,}")
print(f"Số patientunitstayid : {n_unique:,}")
print(f"Số trùng lặp         : {n_dup}")
print(f"{'✅' if n_dup == 0 else '⚠️'} Khóa chính duy nhất")

n_bad_id = (df_vital["patientunitstayid"] <= 0).sum()
print(f"{'✅' if n_bad_id == 0 else '⚠️'} patientunitstayid <= 0: {n_bad_id} dòng")

KIỂM TRA RIÊNG: KHÓA CHÍNH

Số dòng              : 2,240
Số patientunitstayid : 2,240
Số trùng lặp         : 0
✅ Khóa chính duy nhất
✅ patientunitstayid <= 0: 0 dòng


In [4]:
# ============================================================
# KIỂM TRA RIÊNG: Tỷ lệ NaN của 9 đặc trưng
# ============================================================
# Đây là kiểm tra quan trọng nhất với vital_clean.csv, vì
# một số đặc trưng (temperature, systemicmean) rất thưa dữ liệu.
#
# Đánh giá mức độ thiếu:
#   - < 10%  : tốt, dùng được
#   - 10-50% : chấp nhận, XGBoost xử lý
#   - > 50%  : gần như vô dụng, cân nhắc bỏ cột

print("=" * 60)
print("KIỂM TRA RIÊNG: TỶ LỆ NaN CỦA 9 ĐẶC TRƯNG")
print("=" * 60)

feature_cols = [c for c in df_vital.columns if c != "patientunitstayid"]

print(f"\n{'Đặc trưng':20s} {'NaN':>6s} {'Tỷ lệ':>8s}  Đánh giá")
print("-" * 60)
for col in feature_cols:
    n_miss = df_vital[col].isna().sum()
    pct = n_miss / len(df_vital) * 100
    if pct < 10:
        status = "✅ Tốt"
    elif pct < 50:
        status = "ℹ️ Chấp nhận"
    else:
        status = "⚠️ Rất thưa"
    print(f"{col:20s} {n_miss:>6,} {pct:>7.2f}%  {status}")

KIỂM TRA RIÊNG: TỶ LỆ NaN CỦA 9 ĐẶC TRƯNG

Đặc trưng               NaN    Tỷ lệ  Đánh giá
------------------------------------------------------------
heartrate_mean           11    0.49%  ✅ Tốt
heartrate_min            11    0.49%  ✅ Tốt
heartrate_max            11    0.49%  ✅ Tốt
systemicmean_max      1,855   82.81%  ⚠️ Rất thưa
sao2_mean                36    1.61%  ✅ Tốt
sao2_min                 36    1.61%  ✅ Tốt
respiration_mean        205    9.15%  ✅ Tốt
respiration_max         205    9.15%  ✅ Tốt
temperature_mean      2,095   93.53%  ⚠️ Rất thưa


In [5]:
# ============================================================
# KIỂM TRA RIÊNG: Phân bố & outlier
# ============================================================
# Với mỗi đặc trưng, kiểm tra:
#   - Giá trị min/max có nằm trong ngưỡng sinh lý không
#   - Có outlier cực đoan không (dù hợp lệ nhưng hiếm)

print("=" * 60)
print("KIỂM TRA RIÊNG: PHÂN BỐ & OUTLIER")
print("=" * 60)

for col in feature_cols:
    s = df_vital[col].dropna()
    if len(s) == 0:
        print(f"\n{col}: toàn bộ NaN")
        continue
    print(f"\n--- {col} ---")
    print(f"  count={len(s):,}  mean={s.mean():.2f}  std={s.std():.2f}")
    print(f"  min={s.min():.2f}  q25={s.quantile(.25):.2f}  "
          f"median={s.median():.2f}  q75={s.quantile(.75):.2f}  max={s.max():.2f}")
    # Đếm outlier theo IQR
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    n_outlier = ((s < q1 - 3 * iqr) | (s > q3 + 3 * iqr)).sum()
    if n_outlier > 0:
        print(f"  ℹ️ Outlier (ngoài 3×IQR): {n_outlier} dòng "
              f"({n_outlier/len(s)*100:.2f}%) — XGBoost tự xử lý")

KIỂM TRA RIÊNG: PHÂN BỐ & OUTLIER

--- heartrate_mean ---
  count=2,229  mean=84.15  std=16.84
  min=37.00  q25=71.93  median=82.60  q75=95.77  max=147.53

--- heartrate_min ---
  count=2,229  mean=68.82  std=16.29
  min=2.00  q25=58.00  median=68.00  q75=79.00  max=127.00

--- heartrate_max ---
  count=2,229  mean=107.61  std=22.38
  min=47.00  q25=92.00  median=105.00  q75=122.00  max=202.00

--- systemicmean_max ---
  count=385  mean=144.12  std=68.98
  min=21.00  q25=98.00  median=116.00  q75=170.00  max=346.00

--- sao2_mean ---
  count=2,204  mean=96.49  std=2.57
  min=53.50  q25=95.20  median=96.75  q75=98.17  max=100.00
  ℹ️ Outlier (ngoài 3×IQR): 8 dòng (0.36%) — XGBoost tự xử lý

--- sao2_min ---
  count=2,204  mean=86.09  std=12.01
  min=1.00  q25=83.00  median=90.00  q75=93.00  max=100.00
  ℹ️ Outlier (ngoài 3×IQR): 51 dòng (2.31%) — XGBoost tự xử lý

--- respiration_mean ---
  count=2,035  mean=19.67  std=4.85
  min=5.00  q25=16.45  median=18.85  q75=22.05  max=67.03
  ℹ️ 

In [6]:
# ============================================================
# KIỂM TRA RIÊNG: So sánh cohort — ai thiếu vital?
# ============================================================
# Merge patient_clean với vital_clean để xem:
#   - Bao nhiêu bệnh nhân trong cohort KHÔNG có trong vital
#   - Nhóm thiếu vital hoàn toàn có tử vong khác biệt không
#   - Nhóm thiếu temperature/systemicmean có tử vong khác biệt không

print("=" * 60)
print("KIỂM TRA RIÊNG: SO SÁNH COHORT")
print("=" * 60)

df_check = df_patient.merge(df_vital, on="patientunitstayid", how="left")
df_check["vital_missing"] = df_check["heartrate_mean"].isna()

print(f"\nSố bệnh nhân trong cohort     : {len(df_patient):,}")
print(f"Số bệnh nhân có vital         : {(~df_check['vital_missing']).sum():,}")
print(f"Số bệnh nhân THIẾU hoàn toàn  : {df_check['vital_missing'].sum():,}")

# So sánh tỷ lệ tử vong: có vital vs thiếu vital
print("\nSo sánh tỷ lệ tử vong: có vital vs thiếu vital")
result = df_check.groupby("vital_missing")["mortality"].agg(
    count="count", deaths="sum", mortality_rate="mean"
).round(4)
result.index = ["Có vital", "Thiếu vital"]
print(result.to_string())

# So sánh nhóm thiếu temperature và systemicmean
for col in ["temperature_mean", "systemicmean_max"]:
    print(f"\nSo sánh tỷ lệ tử vong: có vs thiếu {col}")
    df_check[f"{col}_missing"] = df_check[col].isna()
    result = df_check.groupby(f"{col}_missing")["mortality"].agg(
        count="count", deaths="sum", mortality_rate="mean"
    ).round(4)
    result.index = [f"Có {col}", f"Thiếu {col}"]
    print(result.to_string())

KIỂM TRA RIÊNG: SO SÁNH COHORT

Số bệnh nhân trong cohort     : 2,326
Số bệnh nhân có vital         : 2,229
Số bệnh nhân THIẾU hoàn toàn  : 97

So sánh tỷ lệ tử vong: có vital vs thiếu vital
             count  deaths  mortality_rate
Có vital      2229     186          0.0834
Thiếu vital     97       5          0.0515

So sánh tỷ lệ tử vong: có vs thiếu temperature_mean
                        count  deaths  mortality_rate
Có temperature_mean       145      24          0.1655
Thiếu temperature_mean   2181     167          0.0766

So sánh tỷ lệ tử vong: có vs thiếu systemicmean_max
                        count  deaths  mortality_rate
Có systemicmean_max       385      48          0.1247
Thiếu systemicmean_max   1941     143          0.0737


In [7]:
# ============================================================
# KIỂM TRA RIÊNG: Tương quan giữa 9 đặc trưng
# ============================================================
# Tương quan quá cao (> 0.9) giữa 2 đặc trưng → dư thừa.
# Ví dụ: heartrate_mean và heartrate_min có thể tương quan mạnh.

print("=" * 60)
print("KIỂM TRA RIÊNG: TƯƠNG QUAN")
print("=" * 60)

corr = df_vital[feature_cols].corr().round(2)
print("\nMa trận tương quan (chỉ hiển thị |r| > 0.5):")
print("-" * 60)

found = False
for i in range(len(feature_cols)):
    for j in range(i + 1, len(feature_cols)):
        c1, c2 = feature_cols[i], feature_cols[j]
        r = corr.loc[c1, c2]
        if abs(r) > 0.5:
            print(f"  {c1:20s} ↔ {c2:20s}: r = {r:+.2f}")
            found = True
if not found:
    print("  ✅ Không có cặp tương quan cao (|r| > 0.5)")

KIỂM TRA RIÊNG: TƯƠNG QUAN

Ma trận tương quan (chỉ hiển thị |r| > 0.5):
------------------------------------------------------------
  heartrate_mean       ↔ heartrate_min       : r = +0.81
  heartrate_mean       ↔ heartrate_max       : r = +0.78
  heartrate_min        ↔ heartrate_max       : r = +0.54
  respiration_mean     ↔ respiration_max     : r = +0.53


In [8]:
# ============================================================
# KẾT LUẬN: Checklist chất lượng vital_clean.csv
# ============================================================

print("=" * 60)
print("CHECKLIST CHẤT LƯỢNG: vital_clean.csv")
print("=" * 60)

def check_range(series, lo, hi):
    s = series.dropna()
    return ((s >= lo) & (s <= hi)).all() if len(s) > 0 else True

checks = {
    "Khóa chính duy nhất":        df_vital["patientunitstayid"].duplicated().sum() == 0,
    "Không có dòng trùng hoàn toàn": df_vital.duplicated().sum() == 0,
    "patientunitstayid > 0":      (df_vital["patientunitstayid"] > 0).all(),
    "heartrate trong [20, 250]":  check_range(df_vital["heartrate_mean"], 20, 250),
    "sao2 trong [50, 100]":       check_range(df_vital["sao2_mean"], 50, 100),
    "respiration trong [5, 80]":  check_range(df_vital["respiration_mean"], 5, 80),
    "temperature trong [30, 45]": check_range(df_vital["temperature_mean"], 30, 45),
    "Tỷ lệ NaN heartrate < 5%":   df_vital["heartrate_mean"].isna().mean() < 0.05,
    "Tỷ lệ NaN sao2 < 5%":        df_vital["sao2_mean"].isna().mean() < 0.05,
    "Tỷ lệ NaN respiration < 15%": df_vital["respiration_mean"].isna().mean() < 0.15,
}

for name, passed in checks.items():
    print(f"{'✅' if passed else '❌'} {name}")

n_pass  = sum(checks.values())
n_total = len(checks)
print(f"\nKết quả: {n_pass}/{n_total} kiểm tra đạt")

if n_pass == n_total:
    print("\n🎉 DỮ LIỆU VITAL SẴN SÀNG cho bước merge!")
else:
    print(f"\n⚠️ Còn {n_total - n_pass} vấn đề cần xem xét.")

CHECKLIST CHẤT LƯỢNG: vital_clean.csv
✅ Khóa chính duy nhất
✅ Không có dòng trùng hoàn toàn
✅ patientunitstayid > 0
✅ heartrate trong [20, 250]
✅ sao2 trong [50, 100]
✅ respiration trong [5, 80]
✅ temperature trong [30, 45]
✅ Tỷ lệ NaN heartrate < 5%
✅ Tỷ lệ NaN sao2 < 5%
✅ Tỷ lệ NaN respiration < 15%

Kết quả: 10/10 kiểm tra đạt

🎉 DỮ LIỆU VITAL SẴN SÀNG cho bước merge!
